# Genoxus Annotation Turorial

Please reference https://genoxuslabs.com/genoxusannotation/ for introduction to our data set. 

# AWS Athena Query

A developer may use AWS Glue to scan our data and generate a table for SQL query with AWS Athena. The following code assumes the developer has successfully scanned the data, and created the following:

database name: **genoxus-data-04-02-2026**

table name: **genoxus_annotation**

A simple SQL query would be the following: 

```
SELECT host_genes 
FROM "genoxus-data-04-02-2026"."genoxus_annotation"
CROSS JOIN UNNEST (host_genes) as t(gene) 
WHERE GRCh='37'
    AND chr='1' 
    AND bp='1-1000000' 
    AND gene.OMIM = '619262'
limit 200;
```

One of the results returned would be:

```
[
   {
      symbol=KLHL17,
      full_name=kelch like family member 17,
      id=339451,
      omim=619262,
      "location="{
         chromosome=1,
         cytogenetic_location=1p36.33,
         start=895966,
         stop=901098,
         "strand=+",
         target_length=5133
      },
      "haploinsufficiency_assertion=null",
      "triplosensitivity_assertion=null"
   },
   {
      symbol=NOC2L,
      full_name=NOC2 like nucleolar associated transcriptional repressor,
      id=26155,
      omim=610770,
      "location="{
         chromosome=1,
         cytogenetic_location=1p36.33,
         start=879582,
         stop=894678,
         "strand=-",
         target_length=15097
      },
      "haploinsufficiency_assertion=null",
      "triplosensitivity_assertion=null"
   },
   {
      symbol=PLEKHN1,
      full_name=pleckstrin homology domain containing N1,
      id=84069,
      "omim=null",
      "location="{
         chromosome=1,
         cytogenetic_location=1p36.33,
         start=901876,
         stop=910487,
         "strand=+",
         target_length=8612
      },
      "haploinsufficiency_assertion=null",
      "triplosensitivity_assertion=null"
   },
   {
      symbol=SAMD11,
      full_name=sterile alpha motif domain containing 11,
      id=148398,
      omim=616765,
      "location="{
         chromosome=1,
         cytogenetic_location=1p36.33,
         start=861120,
         stop=879960,
         "strand=+",
         target_length=18841
      },
      "haploinsufficiency_assertion=null",
      "triplosensitivity_assertion=null"
   }
]
```

# Query Parquet File

To run a simple query against a downloaded Parquet file, a developer may reference the following sample code in Python.


In [ ]:
import duckdb    
parquet_file = "clinvar.parquet"
conn = duckdb.connect(database=':memory:')

chromosome = 1
start_pos = 895966
stop_pos = 901098
var_chunk = stop_pos - start_pos
if var_chunk <= 0:
    conn.close()
    print ("Bad variation info, exit...")
    
else:
    print(f"--- Searching for records where the first assembly's Chromosome is {chromosome} ---")

    # DuckDB uses 1-based indexing for lists/arrays.
    # We also check if the list is not empty to prevent errors.
    query = f"""
        SELECT *
        FROM "{parquet_file}"
        CROSS JOIN UNNEST(Host_Genes) AS t(gene)
        WHERE gene.location.start = {start_pos} 
        AND gene.location.stop = {stop_pos};
    """    

    print (query)

    # Using fetch_df() is often more convenient for inspection
    results = conn.execute(query).fetchall()

    print (f"Found {len(results)} records")
    for r in results:
        print (r)

    conn.close()